# Event-Study DiD and Staggered Rollouts

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


## Dynamic effects: the event-study specification

Replace the single `D` with a set of leads and lags relative to launch:

$$
Y_{it} = \alpha_i + \lambda_t + \sum_{k \neq -1} \tau_k \, \mathbf{1}[t - t^*_i = k] \cdot \text{treated}_i + \varepsilon_{it}
$$

The omitted period $k=-1$ (the week before launch) is the reference. Lead coefficients ($k<-1$) should be ≈ 0 — a visual pre-trend check. Lag coefficients trace the effect's dynamics: ramp-up, peak, decay.

In [ ]:
df = panel.copy()
df["rel"] = np.where(df.treated == 1, df.week - 60, np.nan)
# bin distant leads/lags to stabilize estimates
df["rel_b"] = df.rel.clip(-12, 20)
dummies = pd.get_dummies(df.rel_b, prefix="k", dtype=float)
dummies = dummies.drop(columns=["k_-1.0"])      # reference period
df = pd.concat([df, dummies], axis=1).fillna(0)

pdf = df.set_index(["store", "week"])
kcols = [c for c in df.columns if c.startswith("k_")]
formula = "revenue ~ " + " + ".join([f"`{c}`" for c in kcols]) + \
          " + EntityEffects + TimeEffects"
res = PanelOLS.from_formula(formula, data=pdf).fit(
    cov_type="clustered", clusters=pdf["market"])

est = pd.DataFrame({
    "k": [float(c.split("_")[1]) for c in kcols],
    "tau": [res.params[c] for c in kcols],
    "se": [res.std_errors[c] for c in kcols],
}).sort_values("k")

plt.figure(figsize=(10, 4))
plt.errorbar(est.k, est.tau, yerr=1.96 * est.se, fmt="o", capsize=3)
plt.axvline(-0.5, color="red", ls="--"); plt.axhline(0, color="gray")
plt.xlabel("Weeks relative to launch"); plt.ylabel("Effect on revenue ($k)")
plt.title("Event-study: dynamic effect of Fresh Local program")
plt.tight_layout(); plt.show()

You should see flat leads near zero, then a ramp to ≈ +8 over ~6 weeks — exactly the truth we built in. For stakeholders, this single plot is often the most persuasive artifact an evaluation produces: it shows no pre-divergence *and* the effect's trajectory.

## Staggered adoption: where classic TWFE breaks

Real rollouts are usually staggered: markets adopt in waves. A major econometrics literature (Goodman-Bacon 2021; Callaway & Sant'Anna 2021; Sun & Abraham 2021; de Chaisemartin & D'Haultfœuille 2020) showed that with staggered timing **and** effects that vary over time or across cohorts, the simple TWFE coefficient is a weighted average of many 2×2 comparisons — including "forbidden" ones that use *already-treated* units as controls. Weights can even be negative, and the estimate can have the wrong sign.

Rules of thumb:

- Single adoption date → classic TWFE is fine.
- Staggered adoption with possibly dynamic effects → use a heterogeneity-robust estimator.

**Callaway–Sant'Anna in Python** via the `csdid` package (a port of the R original), or `differences`:

In [ ]:
%pip install csdid
from csdid.att_gt import ATTgt

stag = make_panel(treat_week=50, seed=7)            # cohort A
stag2 = make_panel(treat_markets=(4, 5), treat_week=70, seed=7)
# (In practice: build one panel with a `first_treated` column per market;
#  code below sketches the estimator call.)
stag["first_treated"] = np.where(stag.treated == 1, 50, 0)

att = ATTgt(yname="revenue", tname="week", idname="store",
            gname="first_treated", data=stag)
res_cs = att.fit(est_method="dr")        # doubly robust group-time ATTs
res_cs.aggte(typec="dynamic")            # event-study aggregation

The Callaway–Sant'Anna estimator computes a clean ATT for each (cohort g, time t) cell using only never-treated or not-yet-treated units as controls, then aggregates — by event time (dynamic), by cohort, or overall. If you only remember one thing about staggered DiD: **never let already-treated units serve as controls when effects are dynamic.**